In [ ]:
import os
import sys 
sys.path.append("/workspaces/dev/modules:/workspaces/dev/app")

In [ ]:
from RTWhisper import TokenStreamer, Hyperparameters
from RTWhisper.data import Param
import librosa
import numpy as np
from dotenv import load_dotenv
from pyannote.audio import Inference
import torch
from scipy.spatial.distance import cosine

In [ ]:
load_dotenv()

In [ ]:
HF_TOKEN=os.getenv("HF_TOKEN")
SAMPLE_RATE = 16000

In [ ]:
full_audio, sr = librosa.load("/workspaces/dev/.data/boda.wav", sr=SAMPLE_RATE)

In [ ]:
total_samples = len(full_audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=48000, scale=400))
  rand_len = np.clip(rand_len, 46000, 50000)
  end = min(pos + rand_len, total_samples)

  chunk = full_audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
HYPERPARAMETERS = {
  "sentence_max_prev_sentence": 1,
  "weighted_and_offset_token_boundary": 8000,
  "duration_filter_z": {
    "default": 2.0,
    "ko": 2.0,
    "en": 2.0,
  },
  "probability_filter": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "selector": {
    "search_range_sc": {
      "default": 24000,
      "ko": 24000,
      "en": 24000,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "padding": {
      "default": 3200,
      "ko": 3200,
      "en": 3200,
    },
    "tolerance": {
      "default": 8000,
      "ko": 8000,
      "en": 8000,
    },
  },
  "classifier_max_prev_sc": {
    "default": 96000
  }
}

In [ ]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [ ]:
whisper_service = TokenStreamer.get_instance(hyper)

In [ ]:
param = Param()
param.audio = segments[0]
whisper_service.process(param)

In [ ]:
embedding_model = Inference( 
  model="pyannote/embedding",
  window="whole", 
  use_auth_token=HF_TOKEN,
  device=torch.device("cuda")
)

In [ ]:
mr_ujmj = [(148061, 339589), (338760, 436145), (433746, 517335)]
mr_jyj = [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)]
mr_kbj = [(2563542, 2662059), (4373060, 4438192), (4570430, 4625790) ]
mr_kub = [(880324, 938541), (7042377, 7096457), (7106339, 7156845)]
ms_bih = [(944284, 1009464), (1802918, 1921616), (1921626, 2053013)]

In [ ]:
def get_speaker_embeddings(timestamps):
  embeddings = []
  for t in timestamps:
    a = full_audio[t[0]:t[1]]
    waveform = torch.from_numpy(a).unsqueeze(0)
    embedding = embedding_model({"waveform": waveform, "sample_rate": SAMPLE_RATE})
    embeddings.append(embedding)
  return embeddings

In [ ]:
mr_ujmj = get_speaker_embeddings(mr_ujmj)
mr_jyj = get_speaker_embeddings(mr_jyj)
mr_kbj = get_speaker_embeddings(mr_kbj)
mr_kub = get_speaker_embeddings(mr_kub)
ms_bih = get_speaker_embeddings(ms_bih)
all_embedding = [*mr_ujmj, *mr_jyj, *mr_kbj, *mr_kub, *ms_bih]

In [ ]:
refer_dict = {
  "mr_ujmj": mr_ujmj,
  "mr_jyj": mr_jyj,
  "mr_kbj": mr_kbj,
  "mr_kub": mr_kub,
  "ms_bih": ms_bih
}

In [ ]:
print(refer_dict["mr_ujmj"][0])

In [ ]:
from util import FixedBufferClustering
fixed_buffer_clustering = FixedBufferClustering(refer_dict, MAX_SIZE=10)

In [ ]:
raise Exception("stop")

In [ ]:
from IPython.display import Audio

In [ ]:
segment_id = 0
completed = {}
param = Param()

In [ ]:
target_id = -1

In [ ]:
while segment_id < len(segments):
  segment = segments[segment_id]
  segment_id += 1

  param.audio = segment

  result = whisper_service.process(param)
  completed.update(result.completed)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  param.update(result)


  if result.completed:
    print("문장 및 구간 음성" + "-" * 20)
    for k, v in result.completed.items():
      start = v.tokens[0].start
      end = v.tokens[-1].end
      a = full_audio[start:end]
      waveform = torch.from_numpy(a).unsqueeze(0)
      embedding = embedding_model({"waveform": waveform, "sample_rate": SAMPLE_RATE})
      key = fixed_buffer_clustering.add(embedding)
      print(f"{k} : {v.text}")
      print(f"\t {start /16000} ~ {end/16000}")
      print(key)
    break
  
  if segment_id == target_id:
    break
    
Audio(result.processed_audio, rate=SAMPLE_RATE)


In [ ]:
index = 0
rt = list(result.completed.items())
k, v = rt[index]
start = v.tokens[0].start
end = v.tokens[-1].end
print(f"{k} : {v.text}")
print(f"\t {start} ~ {end}, {start/SAMPLE_RATE/60} ~ {end/SAMPLE_RATE/60}")
Audio(full_audio[start:end], rate=SAMPLE_RATE)

In [ ]:
all_audio = [*mr_ujmj, *mr_jyj, *mr_kbj, *mr_kub, *ms_bih]
audio_idx = 0

In [ ]:
a = all_audio[audio_idx]
audio_idx += 1
Audio(full_audio[a[0]:a[1]], rate=SAMPLE_RATE)

In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [ ]:
for segment in segments:
  word_params.audio = segment

  (result, audio) = whisper_service.transcribe(word_params)
  completed.update(result.completed_dict)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  word_params.order = result.order
  word_params.time_offset = result.time_offset
  word_params.prev_audio = result.prev_audio
  word_params.prev_words = result.prev_words
  word_params.prev_recog = result.prev_recog
  word_params.prev_prob_mean = result.prev_prob_mean
  word_params.prev_prob_std = result.prev_prob_std
  word_params.prev_prob_count = result.prev_prob_count
  word_params.prev_dura_mean = result.prev_dura_mean
  word_params.prev_dura_std = result.prev_dura_std
  word_params.prev_dura_count = result.prev_dura_count

In [ ]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_words:
  if v.is_word: print(v.text)
# for v in prev_recog:
#   print(v.text)